Empezaremos ahora con el data cleaning del raw data set

In [31]:
import pandas as pd
import numpy as np

df = pd.read_csv('/content/ecommerce_raw_dataset.csv')


In [32]:
print (df.head(10))

    order_id customer_id    customer_name               customer_email   age  \
0  ORD-01496   CUST-0189     Arthur Ellis       fgutierrez@example.net  64.0   
1  ORD-00544   CUST-0275    Michael Green                          NaN  31.0   
2  ORD-01269   CUST-0229  Kimberly Dudley                          NaN  30.0   
3  ORD-00529   CUST-0115      Anna Reilly         robert33@example.org  60.0   
4  ORD-01095   CUST-0352    Robert Hanson  patriciagilbert@example.org  41.0   
5  ORD-00808   CUST-0010   Audrey Terrell         hallison@example.com  20.0   
6  ORD-01377   CUST-0255   Arthur Shields         amanda81@example.org  33.0   
7  ORD-00603   CUST-0195     Steven Chang            iwest@example.org  26.0   
8  ORD-01204   CUST-0227    Alan Fletcher       fordkendra@example.org  63.0   
9  ORD-01175   CUST-0254      Jason Pitts           juan68@example.com  28.0   

   gender    country  order_date product_id           product_name  \
0    Male     Mexico  2023-06-01   PRD-3413      

¿Que tendremos que arrelgar?

-   columnas duplicadas
-   Valores null
- inconsitencias en datos
- formatos mixtos
- valores negativos
- inconsistencias en nombres de países
- outliers en paises



In [33]:
# --- 1. Remove duplicates ---
print('before dedup:', df.shape)
df = df.drop_duplicates(subset='order_id', keep='first')
print('After dedup:', df.shape)


before dedup: (1545, 19)
After dedup: (1500, 19)


In [34]:
# --- 2. Standardize text columns ---
df['category'] = df['category'].str.strip().str.title()
df['gender'] = df['gender'].str.strip().str.title()
df['payment_method'] = df['payment_method'].str.strip().str.title()
df['order_status'] = df['order_status'].str.strip().str.title()


In [35]:

# --- 3. Standardize country names ---
country_map = {
    'United States': 'USA', 'Col.': 'Colombia', 'MX': 'Mexico'
}
df['country'] = df['country'].replace(country_map)


In [36]:
# --- 4. Fix date formats ---
df['order_date'] = pd.to_datetime(df['order_date'], dayfirst=False,
                                   infer_datetime_format=True, errors='coerce')


/tmp/ipykernel_243/734029405.py:2: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  df['order_date'] = pd.to_datetime(df['order_date'], dayfirst=False,


In [37]:
# --- 5. Fix negative quantities ---
df['quantity'] = df['quantity'].abs()


In [38]:
# --- 6. Fix age outliers (valid range: 16-100) ---
df['age'] = df['age'].apply(lambda x: np.nan if (x < 16 or x > 100) else x)


In [39]:
# --- 7. Fix price outliers (cap at 99th percentile) ---
p99 = df['unit_price'].quantile(0.99)
df['unit_price'] = df['unit_price'].clip(upper=p99)


In [40]:
# --- 8. Recalculate total_amount after price fix ---
df['total_amount'] = df['unit_price'] * df['quantity'] * (1 - df['discount_pct']/100)


In [44]:
# --- 9. Fill nulls ---
df['shipping_method'] = df['shipping_method'].fillna('Unknown')
df['customer_email'] = df['customer_email'].fillna('Unknown')
df['order_status'] = df['order_status'].fillna('Unknown')
df['age'] = df['age'].fillna(df['age'].median())
# rating: keep nulls — they mean 'did not rate' (informative)


In [45]:
# --- 10. Save clean file ---
df.to_csv('/content/CLEANecommerce_raw_dataset.csv', index=False)
print('Clean shape:', df.shape)


Clean shape: (1500, 19)


In [46]:
print (df.head(10))

    order_id customer_id    customer_name               customer_email   age  \
0  ORD-01496   CUST-0189     Arthur Ellis       fgutierrez@example.net  64.0   
1  ORD-00544   CUST-0275    Michael Green                      Unknown  31.0   
2  ORD-01269   CUST-0229  Kimberly Dudley                      Unknown  30.0   
3  ORD-00529   CUST-0115      Anna Reilly         robert33@example.org  60.0   
4  ORD-01095   CUST-0352    Robert Hanson  patriciagilbert@example.org  41.0   
5  ORD-00808   CUST-0010   Audrey Terrell         hallison@example.com  20.0   
6  ORD-01377   CUST-0255   Arthur Shields         amanda81@example.org  33.0   
7  ORD-00603   CUST-0195     Steven Chang            iwest@example.org  26.0   
8  ORD-01204   CUST-0227    Alan Fletcher       fordkendra@example.org  63.0   
9  ORD-01175   CUST-0254      Jason Pitts           juan68@example.com  28.0   

   gender    country order_date product_id           product_name  \
0    Male     Mexico 2023-06-01   PRD-3413        